# AgentsVille Trip Planner - Project Assignment

In this project, you'll implement an AI system to help you plan a trip--to the wonderful city of AgentsVille!

Your AI system will demonstrate advanced LLM reasoning techniques including:

1. **Role-Based Prompting** - Your agent will act as a specialized travel planner
2. **Chain-of-Thought Reasoning** - Step-by-step planning of itineraries
3. **ReAct Prompting** - Thought → Action → Observation cycles
4. **Feedback Loops** - Self-evaluation using tools in the ReAct loop to find mistakes and improve plans

You'll simulate external API calls to gather weather data and activities. Then, process this information to create personalized travel itineraries based on interests and other constraints. Last, you'll implement a feedback loop to refine the plan.

Your task is to build a travel agent that can plan the perfect AgentsVille vacation!

## Project Instructions

Here are the steps you'll follow:

1. **Define Vacation Details**
    - Specify the trip duration, interests, and constraints.
    - Use Pydantic to structure and verify this information in a class called `VacationInfo`.
2. **Review Weather and Activity schedules**
    - Simulate API calls to gather weather data and available activities in bulk
    - Review the data manually to understand the available options
3. **The ItineraryAgent**
    - Implement an agent that generates a day-by-day itinerary based on the vacation details
    - The system prompt will guide the agent's reasoning through a step-by-step planning process to take travel preferences (e.g. destination, dates, interests) and generate a detailed day-by-day itinerary
    - Craft the components of the prompt (including the role, task/instructions, output format, examples, and context) to elicit the best possible itinerary in one LLM call
4. **Evaluating the Itinerary**
    - Evaluate the itinerary using a set of criteria to ensure a high-quality travel plan
        - For instance, does the itinerary match the city and the dates requested?
        - Or, is the total cost calulation accurate and is it within budget?
        - Or, does the agent hallucinate any activities that are not available?
        - Or, does the agent suggest activities that are not suitable for the weather? This specific evaluation function will require the use of an LLM to compare the event description against the weather data.
5. **Defining the Tools**
    - We will define four tools to assist the agent
        - **calculator_tool**: to accurately calculate costs
        - **get_activities_by_date_tool**: to retrieve activities for a specific date
        - **run_evals_tool**: to evaluate the itinerary against the criteria
        - **final_answer_tool**: to provide the final answer in a structured format
6. **The ItineraryRevisionAgent**
    - We will implement a second agent that revises the itinerary based on feedback using the ReAct THOUGHT → ACTION → OBSERVATION cycle.
        - The LLM will first generated a THOUGHT / ACTION message, which contains reasoning steps and a tool call invocation.
        - The Python code will parse the tool call and execute it, returning the result as a string to the LLM in an OBSERVATION message.
        - After this cycle repeats n number of times, the LLM will invoke the final_answer_tool to signal to the Python code to end the loop and return the final answer.
    - This agent will also **incorporate feedback on the initial itinerary** from the travelers to ensure the final plan has **at least 2 activities per day**. A new evaluation function using a powerful LLM will be created to check this user feedback.
    - The agent will use the tools above to refine the plan iteratively, checking the weather and available activities, and ensuring the itinerary meets all constraints.
7. **Something just for fun!**
    - To wrap things up we'll create a fun, narrative summary of the trip, highlighting the best activities and experiences!

## Initial Setup

Let's start with settin up our environment and defining the vacation details.

In [1]:
# Local setup: launch JupyterLab from the repository root using uv.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_lib.py").is_file():
    raise RuntimeError("Launch JupyterLab from the project root: uv run --locked jupyter lab")
print(f"Python {sys.version.split()[0]} — local project environment")


Python 3.12.3 — local project environment


In [2]:
# Dependencies are installed and locked outside the notebook.
# From Git Bash, run: bash scripts/setup.sh
# This cell does not install or change packages.
import pandas as pd
import pydantic
from openai import OpenAI
print("Notebook dependencies imported successfully.")


Notebook dependencies imported successfully.


In [3]:
# Load the API key from .env without embedding or displaying it here.
# Constructing the client does not make an API request.
from local_setup import create_client

client = create_client()
print("API client configured. No request has been sent.")


API client configured. No request has been sent.

In [4]:
# Throughout this project you can experiment with different OpenAI models.
# By default we will use GPT-4.1-mini, which is a good balance of speed and cost.
from enum import Enum

class OpenAIModel(str, Enum):
    GPT_41 = "gpt-4.1"  # Strong default choice for development tasks, particularly those requiring speed, responsiveness, and general-purpose reasoning. 
    GPT_41_MINI = "gpt-4.1-mini"  # Fast and affordable, good for brainstorming, drafting, and tasks that don't require the full power of GPT-4.1.
    GPT_41_NANO = "gpt-4.1-nano"  # Retired; retained only as a record of the starter model option.

MODEL = OpenAIModel.GPT_41_MINI  # Default model for this project


## Define Vacation Details

Let's encode the details of our vacation in JSON format and verify it using Pydantic.

In practice, a chatbot agent could gather the information of a user. After it has gathered all the information it needs, it could generate this JSON object from the chat transcript. We will skip that step to focus on the itinerary generation itself.

In [5]:
# The Vacation Info Data Structure
# No changes needed here, but you may choose to personalize the data.

VACATION_INFO_DICT = {
    "travelers": [
        {
            "name": "Yuri",
            "age": 30,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, movies,
            # music, photography, reading, sports, technology, theatre, tennis, writing
            "interests": ["tennis", "cooking", "comedy", "technology"],
        },
        {
            "name": "Hiro",
            "age": 25,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, movies,
            # music, photography, reading, sports, technology, theatre, tennis, writing
            "interests": ["reading", "music", "theatre", "art"],
        },
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",  # Mock data exists for 2025-06-10
    "date_of_departure": "2025-06-12",  # ...until 2025-06-15.
    "budget": 130,  # Budget is in fictional currency units.
}

In [6]:
# Validate trip inputs before retrieving data or making model requests.
from project_lib import Interest
from typing import List
from pydantic import BaseModel, ConfigDict, Field, model_validator
import datetime
from pprint import pprint


class Traveler(BaseModel):
    """A traveler with a name, age, and at least one supported interest."""

    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")

    name: str = Field(min_length=1)
    age: int = Field(ge=0, strict=True)
    interests: List[Interest] = Field(min_length=1)


class VacationInfo(BaseModel):
    """Validated travelers, destination, inclusive dates, and activity budget.

    Dates are parsed from ISO strings. The budget uses whole fictional units.
    Dataset availability is checked during lookup, not imposed on this model.
    """

    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")

    travelers: List[Traveler] = Field(min_length=1)
    destination: str = Field(min_length=1)
    date_of_arrival: datetime.date
    date_of_departure: datetime.date
    budget: int = Field(ge=0, strict=True)

    @model_validator(mode="after")
    def validate_date_order(self) -> "VacationInfo":
        if self.date_of_departure < self.date_of_arrival:
            raise ValueError("date_of_departure must be on or after date_of_arrival")
        return self


# Validate the VacationInfo data structure
vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)

# Display the vacation info as a dictionary
pprint(vacation_info.model_dump())

# Check that VacationInfo contains the expected data
assert "travelers" in vacation_info.model_dump().keys(), "VacationInfo should contain 'travelers' key"
assert "destination" in vacation_info.model_dump().keys(), "VacationInfo should contain 'destination' key"
assert "date_of_arrival" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_arrival' key"
assert "date_of_departure" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_departure' key"
assert "budget" in vacation_info.model_dump().keys(), "VacationInfo should contain 'budget' key"
assert isinstance(vacation_info.travelers, list), "Travelers should be a list"
assert all(isinstance(traveler, Traveler) for traveler in vacation_info.travelers), "All travelers should be instances of Traveler class"
assert isinstance(vacation_info.date_of_arrival, datetime.date), "date_of_arrival should be a date"
assert isinstance(vacation_info.date_of_departure, datetime.date), "date_of_departure should be a date"
assert isinstance(vacation_info.budget, int), "budget should be an integer"

# If all assertions pass, print a success message
print("✅ VacationInfo data structure is valid!")

{'budget': 130,
 'date_of_arrival': datetime.date(2025, 6, 10),
 'date_of_departure': datetime.date(2025, 6, 12),
 'destination': 'AgentsVille',
 'travelers': [{'age': 30,
                'interests': [tennis, cooking, comedy, technology],
                'name': 'Yuri'},
               {'age': 25,
                'interests': [reading, music, theatre, art],
                'name': 'Hiro'}]}
✅ VacationInfo data structure is valid!


## Review Weather and Activity Schedules

Now that we have the trip details, we can retrieve the weather and activity schedules for the dates of the trip.

We will call an API to get all the data at once, in order to be able to include it in the context for our itinerary planning agent.

Also, we will format the data as Pandas DataFrames for easier viewing. Take the time to read and understand the data to see if the agent notices the same things you do. For instance:
- What does the weather look like for the trip? On what days it is sunny, rainy, or cloudy?
- What activities are available on each day? Are there any special events or festivals related to the user's interests?

In [7]:
# Retrieve weather for every day, including arrival and departure.
from project_lib import call_weather_api_mocked
import pandas as pd

pd.set_option("display.max_colwidth", None)

weather_for_dates = []
for ts in pd.date_range(
    start=vacation_info.date_of_arrival,
    end=vacation_info.date_of_departure,
    freq="D",
    inclusive="both",
):
    trip_date = ts.strftime("%Y-%m-%d")
    forecast = call_weather_api_mocked(date=trip_date, city=vacation_info.destination)
    if not forecast:
        raise ValueError(
            f"No weather data for {vacation_info.destination} on {trip_date}. "
            "The supplied data covers AgentsVille from 2025-06-10 through 2025-06-15."
        )
    weather_for_dates.append(forecast)

weather_for_dates_df = pd.DataFrame(weather_for_dates)
weather_for_dates_df


,date,city,temperature,temperature_unit,condition,description
0,2025-06-10,AgentsVille,31,celsius,clear,A bright and sunny day in AgentsVille with clear skies and warm temperatures. Perfect weather for outdoor activities!
1,2025-06-11,AgentsVille,34,celsius,partly cloudy,"A warm day with periods of sunshine and mixed clouds, making it a perfect opportunity for outdoor activities."
2,2025-06-12,AgentsVille,28,celsius,thunderstorm,"A thunderstorm is expected to roll in during the afternoon, bringing heavy rain and gusty winds. The atmosphere will feel charged with humidity, creating a sultry and dramatic setting as clouds build in the sky."


In [8]:
# Retrieve available activities for each validated trip date.
from project_lib import call_activities_api_mocked

activities_for_dates = []
for ts in pd.date_range(
    start=vacation_info.date_of_arrival,
    end=vacation_info.date_of_departure,
    freq="D",
    inclusive="both",
):
    trip_date = ts.strftime("%Y-%m-%d")
    daily_activities = call_activities_api_mocked(date=trip_date, city=vacation_info.destination)
    if not daily_activities:
        raise ValueError(
            f"No activities for {vacation_info.destination} on {trip_date}. "
            "The supplied data covers AgentsVille from 2025-06-10 through 2025-06-15."
        )
    activities_for_dates.extend(daily_activities)

activities_for_dates_df = pd.DataFrame(activities_for_dates)
activities_for_dates_df


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00,2025-06-10 11:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00,2025-06-10 13:30,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"
2,event-2025-06-10-2,Artful Athletics: Paint & Play Extravaganza,2025-06-10 15:00,2025-06-10 17:00,"Creative Courts Park, AgentsVille","Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide! At 'Artful Athletics: Paint & Play Extravaganza', you'll participate in collaborative outdoor murals inspired by your favorite sports, and then get moving with fun, friendly sports mini-games. Whether you love painting or playing, this event celebrates creativity, teamwork, and the joy of movement under the open sky. Perfect for art lovers and sports enthusiasts alike—come ready to express yourself and get active! (Event is held outdoors; in case of rain, we move indoors to the Community Gym nearby.)",15,"[art, sports]"
3,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"
4,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00,2025-06-11 10:30,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
5,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00,2025-06-11 13:30,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI cr

## The ItineraryAgent

First we will review the Pydantic objects used for defining the output of our agent, the TravelPlan, ItineraryDay, Activity, and Weather classes.

Second, we will create a Chain-of-Thought prompt to guide the agent in planning the trip. This prompt will instruct the agent to consider the weather, activities, and user preferences when creating the itinerary.

Third, we will run the agent to produce the TravelPlan object, which will will refine in the following steps.

In [9]:
# Review the data structure we will use for representing a TravelPlan, which includes
# weather, activity_recommendations, and itinerary for each day of the trip.
# Our goal is to take a VacationInfo object and return a TravelPlan object.
# No changes are needed here.

class Weather(BaseModel):
    temperature: float
    temperature_unit: str
    condition: str


class Activity(BaseModel):
    activity_id: str
    name: str
    start_time: datetime.datetime
    end_time: datetime.datetime
    location: str
    description: str
    price: int
    related_interests: List[Interest]


class ActivityRecommendation(BaseModel):
    activity: Activity
    reasons_for_recommendation: List[str]


class ItineraryDay(BaseModel):
    date: datetime.date
    weather: Weather
    activity_recommendations: List[ActivityRecommendation]


class TravelPlan(BaseModel):
    city: str
    start_date: datetime.date
    end_date: datetime.date
    total_cost: int
    itinerary_days: List[ItineraryDay]

In [10]:
# Plan from the supplied trip, weather, and activity records.
import json
from project_lib import ChatAgent
from typing import Optional

ITINERARY_AGENT_SYSTEM_PROMPT = f"""
You are an expert travel planner for AgentsVille. Create a practical daily
itinerary that fits the travelers and the supplied activity calendar.

## Task

Plan the trip using this sequence:
1. Read VacationInfo: identify the city, inclusive arrival/departure dates,
   each traveler's interests, and the total activity budget.
2. For every trip date, inspect that day's weather and available activities.
   Choose at least one activity per day. Include only activities from the
   supplied records and schedule each on its actual date and time.
3. Match the travelers' interests. Across the trip, give every traveler at
   least one activity matching an interest. Explain the matches in
   reasons_for_recommendation. For each interest-match reason, use the form
   "TravelerName: matching_interest_1, matching_interest_2." The named
   interests must appear BOTH in that traveler's interests and the selected
   activity's related_interests. Use the exact interest labels from the data.
   Keep weather and scheduling explanations in separate reason strings.
4. Avoid outdoor-only activities during rain or thunderstorms. An activity
   with an explicit indoor backup in its description is allowed; explain
   that backup in reasons_for_recommendation. Do not invent a backup.
5. Keep activities chronological, with no overlapping times or duplicate
   activity IDs. Copy every selected activity field exactly from its source
   record, including its original location, description, price, and interests.
   Put explanations in reasons_for_recommendation, not in the activity data.
6. Copy each day's temperature, temperature_unit, and condition from its
   forecast. Include every trip day once, in date order.
7. After making the FINAL activity choices, write each day's price addition
   and subtotal in the planning summary. Add those subtotals for total_cost.
   Recompute from the activities actually included in the final JSON; if an
   activity is added or removed, discard the earlier total and calculate it
   again. Count each selected price once. Do not multiply by travelers or
   invent other costs. The final sum must match total_cost exactly and must
   not exceed the budget. Recheck dates, records, weather, and interests.

## Output Format

Return these two sections:

ANALYSIS:
A brief planning summary covering trip dates, interest matches, and weather
choices, followed by one price equation per day and the grand-total equation.
Use the final selected activity prices in these equations.

FINAL OUTPUT:
One JSON object inside a single fenced block labeled json. It must be an
instance of the TravelPlan schema below, including all required nested
fields. Use ISO dates/times and whole-number prices. Return the plan data,
not the schema. Before writing FINAL OUTPUT, sort itinerary_days by date and
sort each day's activity_recommendations by activity.start_time ascending.
Do not add commentary after the JSON block.

<travel_plan_schema>
{json.dumps(TravelPlan.model_json_schema(), indent=2)}
</travel_plan_schema>

## Examples

Interest match: if Alex lists technology and tennis, and an activity is
tagged technology and movies, the interest reason is "Alex: technology."
Only the intersection is a match.

Cost calculation: if selected source activities cost 15 and 20, total_cost
is 35 for the itinerary, including when two travelers attend.
Weather choice: a rainy-day outdoor event with an explicit indoor backup
may be selected; describe that backup in the recommendation while keeping
the original activity record unchanged.

## Context

The following records provide the facts for the plan. The user message
contains the same validated vacation details.

<vacation_info>
{vacation_info.model_dump_json(indent=2)}
</vacation_info>

<weather_records>
{json.dumps(weather_for_dates, indent=2)}
</weather_records>

<activity_records>
{json.dumps(activities_for_dates, indent=2)}
</activity_records>

Use the supplied records and obey the TravelPlan schema. If the constraints
cannot be satisfied, state the conflict rather than inventing activities,
prices, weather, or an itinerary that pretends to meet the constraints.
""".strip()

assert "TASK" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'TASK' section"
assert "OUTPUT FORMAT" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'OUTPUT FORMAT' section"


class ItineraryAgent(ChatAgent):
    """An agent that plans itineraries based on vacation information, weather, and activities."""
    system_prompt = ITINERARY_AGENT_SYSTEM_PROMPT

    def get_itinerary(self, vacation_info: VacationInfo, model: Optional[OpenAIModel] = None) -> TravelPlan:
        """Generates a travel itinerary based on the provided vacation information."""
        from project_lib import print_in_box
        response = (self.chat(
            user_message=vacation_info.model_dump_json(indent=2),
            add_to_messages=False,
            model=model or self.model,
        ) or "").strip()

        print_in_box(response, "Raw Response")

        # Parse the response
        json_text = response.strip()

        if "```json" in json_text:
            json_text = json_text.split("```json")[1].split("```")[0].strip()

        try:
            travel_plan = TravelPlan.model_validate_json(json_text)
            return travel_plan
        except Exception as e:
            print("Error validating the following text as TravelPlan JSON:")
            print(json_text)
            raise

itinerary_agent = ItineraryAgent(client=client, model=MODEL)


╔══════════════════════════════════════════[ ItineraryAgent - System Prompt ]══════════════════════════════════════════╗
║ You are an expert travel planner for AgentsVille. Create a practical daily                                           ║
║ itinerary that fits the travelers and the supplied activity calendar.                                                ║
║ ## Task                                                                                                              ║
║ Plan the trip using this sequence:                                                                                   ║
║ 1. Read VacationInfo: identify the city, inclusive arrival/departure dates,                                          ║
║    each traveler's interests, and the total activity budget.                                                         ║
║ 2. For every trip date, inspect that day's weather and available activities.                                         ║
║    Choose at least one activi

In [11]:
# Generate the travel itinerary
# No changes needed here, though you can change the model to a different one if you want.

travel_plan_1 = itinerary_agent.get_itinerary(
    vacation_info=vacation_info,
    model=MODEL,  # Optionally, you can change the model here
)

if travel_plan_1 is not None:
    print("✅ Initial itinerary generated successfully. Congratulations!")


╔═══════════════════════════════════════════[ ItineraryAgent - User Prompt ]═══════════════════════════════════════════╗
║ {                                                                                                                    ║
║   "travelers": [                                                                                                     ║
║     {                                                                                                                ║
║       "name": "Yuri",                                                                                                ║
║       "age": 30,                                                                                                     ║
║       "interests": [                                                                                                 ║
║         "tennis",                                                                                                    ║
║         "cooking",           


╔═══════════════════════════════════════════════════[ Raw Response ]═══════════════════════════════════════════════════╗
║ ANALYSIS:                                                                                                            ║
║ The trip is from 2025-06-10 to 2025-06-12 in AgentsVille with a total budget of 130. Yuri's interests are tennis,    ║
║ cooking, comedy, and technology. Hiro's interests are reading, music, theatre, and art. Weather is clear on June 10  ║
║ and partly cloudy on June 11, both favorable for outdoor or indoor activities. However, June 12 has a thunderstorm   ║
║ expected in the afternoon, necessitating indoor or activities with indoor backup for that day.                       ║
║ On June 10, three activities are available without weather constraints: the FutureTech Breakfast Meet-Up             ║
║ (technology) suits Yuri, Serve & Savor: Tennis and Taste Luncheon fits Yuri's tennis and cooking interests, and      ║
║ Artful Athletics (art) matche

## Evaluating the Itinerary

We've successfully created an itinerary, but how do we know if it's any good?

Now we will create some evaluation functions (sometimes called evals) to help us determine the quality of the itinerary. We will not only want our final output to be of the highest quality possible initially, but we also want to give the chance for the LLM to reflect on its own output and make improvements at a second pass.

If the itinerary does not meet all the criteria specified here, no worries! Afterwards, we will implement a feedback loop that will allow the agent to improve its plan iteratively.

In [12]:
# Helper functions for running the evaluation functions
# No change needed here.

class AgentError(Exception):
    pass


class EvaluationResults(BaseModel):
    success: bool
    failures: List[str]
    eval_functions: List[str]


def get_eval_results(vacation_info, final_output, eval_functions) -> EvaluationResults:
    """
    Evaluates the final output of the itinerary agent against a set of evaluation functions.
    Args:
        vacation_info (VacationInfo): The vacation information used to generate the itinerary.
        final_output (TravelPlan): The final output from the itinerary agent.
        eval_functions (List[callable]): A list of evaluation functions to apply.
    Returns:
        EvaluationResults: An object containing the success status, any failures, and the names of the evaluation functions used.
    """
    from project_lib import print_in_box
    if not isinstance(vacation_info, VacationInfo):
        raise ValueError("vacation_info must be an instance of VacationInfo")
    if not isinstance(final_output, TravelPlan):
        raise ValueError("final_output must be an instance of TravelPlan")
    if not isinstance(eval_functions, list) or not all(
        callable(fn) for fn in eval_functions
    ):
        raise ValueError("eval_functions must be a list of callable functions")
    eval_results = []
    for eval_fn in eval_functions:
        try:
            eval_fn(vacation_info, final_output)
        except AgentError as e:
            error_msg = str(e)
            print_in_box(error_msg, title="Evaluation Error")
            print("\n\n")

            eval_results.append(error_msg)
    return EvaluationResults(
        success=len(eval_results) == 0,
        failures=eval_results,
        eval_functions=[fn.__name__ for fn in eval_functions],
    )


In [13]:
# Basic evaluation functions
# No changes needed here.

def eval_start_end_dates_match(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the arrival and departure dates in vacation_info match the start and end dates in final_output.

    Args:
        vacation_info (dict): Contains the vacation details including arrival and departure dates
        final_output (dict): Contains the itinerary details including start and end dates

    Raises:
        AgentError: If either the arrival date doesn't match the start date or the departure date doesn't match the end date
    """
    if (
        vacation_info.date_of_arrival != final_output.start_date
        or vacation_info.date_of_departure != final_output.end_date
    ):
        raise AgentError(
            f"Dates do not match: {vacation_info.date_of_arrival} != {final_output.start_date} or {vacation_info.date_of_departure} != {final_output.end_date}"
        )

    if final_output.start_date > final_output.end_date:
        raise AgentError(
            f"Start date is after end date: {final_output.start_date} > {final_output.end_date}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_start_end_dates_match],
)

def eval_daily_schedule_is_valid(vacation_info: VacationInfo, final_output: TravelPlan):
    """Check city, daily coverage, source weather, event placement, and timing."""
    from project_lib import call_weather_api_mocked

    day_count = (vacation_info.date_of_departure - vacation_info.date_of_arrival).days + 1
    expected_dates = [vacation_info.date_of_arrival + datetime.timedelta(days=n) for n in range(day_count)]
    if final_output.city != vacation_info.destination:
        raise AgentError("Itinerary city does not match the vacation destination.")
    if [day.date for day in final_output.itinerary_days] != expected_dates:
        raise AgentError("Include every trip day exactly once, in date order.")
    seen = set()
    for day in final_output.itinerary_days:
        forecast = call_weather_api_mocked(day.date.isoformat(), vacation_info.destination)
        expected_weather = {key: forecast.get(key) for key in ("temperature", "temperature_unit", "condition")}
        if day.weather.model_dump() != expected_weather:
            raise AgentError(f"Weather does not match the supplied forecast on {day.date}.")
        if not day.activity_recommendations:
            raise AgentError(f"No activities scheduled on {day.date}.")
        previous_end = None
        for recommendation in day.activity_recommendations:
            activity = recommendation.activity
            if activity.activity_id in seen:
                raise AgentError(f"Duplicate activity: {activity.activity_id}.")
            seen.add(activity.activity_id)
            if activity.start_time.date() != day.date or activity.end_time.date() != day.date:
                raise AgentError(f"Activity {activity.activity_id} is placed on the wrong day.")
            if activity.start_time >= activity.end_time or (previous_end is not None and activity.start_time < previous_end):
                raise AgentError(f"Activities are out of order or overlap on {day.date}.")
            if not recommendation.reasons_for_recommendation or any(not reason.strip() for reason in recommendation.reasons_for_recommendation):
                raise AgentError(f"Missing recommendation reasons for {activity.activity_id}.")
            for reason in recommendation.reasons_for_recommendation:
                for traveler in vacation_info.travelers:
                    prefix = traveler.name + ": "
                    if reason.startswith(prefix):
                        claimed = set(reason[len(prefix):].rstrip(".").split(", "))
                        matching = set(traveler.interests) & set(activity.related_interests)
                        if not claimed or not claimed <= matching:
                            raise AgentError(
                                f"Unsupported interest claim for {traveler.name} in {activity.activity_id}. "
                                f"Use only these matches: {sorted(str(value) for value in matching)}."
                            )
            previous_end = activity.end_time


In [14]:
# Evaluation functions related to the budget and total cost
# No changes needed here.


def eval_total_cost_is_accurate(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output matches the sum of all activity prices.

    Args:
        vacation_info (dict): Contains the vacation details
        final_output (dict): Contains the itinerary details including activities with prices and total cost

    Raises:
        AgentError: If the calculated total cost doesn't match the stated total cost
    """
    actual_total_cost = 0

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            actual_total_cost += activity_recommendation.activity.price

    stated_total_cost = int(final_output.total_cost)

    if actual_total_cost != stated_total_cost:
        raise AgentError(
            f"Stated total cost does not match calculated total cost: {actual_total_cost} != {stated_total_cost}"
        )
    
def eval_total_cost_is_within_budget(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output is within the budget specified in vacation_info.

    Args:
        vacation_info (dict): Contains the vacation details including budget
        final_output (dict): Contains the itinerary details including total cost

    Raises:
        AgentError: If the total cost exceeds the budget
    """
    stated_total_cost = int(final_output.total_cost)
    if stated_total_cost > vacation_info.budget:
        raise AgentError(
            f"Total cost exceeds budget: {stated_total_cost} > {vacation_info.budget}"
        )

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_total_cost_is_accurate, eval_total_cost_is_within_budget],
)


EvaluationResults(success=True, failures=[], eval_functions=['eval_total_cost_is_accurate', 'eval_total_cost_is_within_budget'])

In [15]:
# The final output contains copies of the activities, so we need to verify that the copies are accurate
# and not hallucinated.
# No changes needed here.

def eval_itinerary_events_match_actual_events(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the events listed in the itinerary match the actual events

    Args:
        vacation_info (dict): Contains the vacation details including traveler information and their interests
        final_output (dict): Contains the itinerary details including daily activities

    Raises:
        AgentError: If any traveler has no matching activities or if one traveler has more than twice
                   the number of matching activities compared to another traveler
    """
    from project_lib import call_activity_by_id_api_mocked
    event_ids_not_matching = []
    event_ids_missing = []

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            event_id = activity_recommendation.activity.activity_id
            # Assuming get_event_by_id is a function that retrieves the event by its ID

            reference_event = call_activity_by_id_api_mocked(event_id)

            if reference_event is None:
                event_ids_missing.append(event_id)

            elif Activity(**reference_event) != activity_recommendation.activity:
                print(
                    "---\n"
                    f"Event ID {event_id} does not match the reference event:\n"
                    f"Reference Event: {reference_event}\n"
                    f"Activity Event: {activity_recommendation.activity.model_dump()}"
                )
                event_ids_not_matching.append(event_id)
            else:
                # The event matches, so we can continue
                pass

    if event_ids_missing or event_ids_not_matching:
        raise AgentError(
            f"Event IDs missing: {event_ids_missing}\nEvent IDs not matching: {event_ids_not_matching}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_events_match_actual_events],
)


EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_events_match_actual_events'])

In [16]:
# Check that the itinerary includes at least one activity matching each traveler's interests.
# No changes needed here.

def eval_itinerary_satisfies_interests(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the itinerary includes activities matching each traveler's interests.

    This function checks that each traveler has at least one activity in the itinerary that matches their interests.

        Args:
        vacation_info (dict): Contains the vacation details including traveler information and their interests
        final_output (dict): Contains the itinerary details including daily activities

    Raises:
        AgentError: If any traveler has no matching activities or if one traveler has more than twice
                   the number of matching activities compared to another traveler
    """
    traveler_to_interests = {}
    traveler_to_interest_hit_counts = {}

    for traveler in vacation_info.travelers:
        traveler_to_interests[traveler.name] = traveler.interests
        traveler_to_interest_hit_counts[traveler.name] = 0

    for traveler_name, interests in traveler_to_interests.items():
        for itinerary_day in final_output.itinerary_days:
            for activity_recommendation in itinerary_day.activity_recommendations:
                # Check if the activity matches any of the traveler's interests
                matching_interests = set(traveler_to_interests[traveler_name]) & set(
                    activity_recommendation.activity.related_interests
                )

                if matching_interests:
                    traveler_to_interest_hit_counts[traveler_name] += 1
                    print(
                        f"✅ Traveler {traveler_name} has a match with interest {matching_interests} at {activity_recommendation.activity.name}"
                    )

    travelers_with_no_interest_hits = [
        traveler
        for traveler, interest_hit_count in traveler_to_interest_hit_counts.items()
        if interest_hit_count == 0
    ]

    # If any of the travelers have 0 matches, raise an error
    if travelers_with_no_interest_hits:
        raise AgentError(
            f"Travelers {travelers_with_no_interest_hits} has no matches with the itinerary."
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_satisfies_interests],
)


✅ Traveler Yuri has a match with interest {tennis, cooking} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Artful Athletics: Paint & Play Extravaganza
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art, music} at AgentsVille Art & Music Fusion Fest
✅ Traveler Hiro has a match with interest {music} at Soundtrack Picnic: Lunchtime Movies & Melodies


EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_satisfies_interests'])

In [17]:
# Judge activity/weather compatibility with the course's LLM evaluator.
ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
You are a travel activity evaluator. Decide whether the supplied activity
can take place under the supplied weather conditions.

## Task

Use the activity description and weather condition to make one judgment:
- An explicitly indoor activity is compatible with rain or thunderstorms.
- An outdoor activity without an indoor backup is incompatible with rain
  or thunderstorms. Clear, sunny, or partly cloudy weather alone is not a
  reason to reject an outdoor activity.
- An activity with an explicit indoor backup for rain is compatible with
  rainy or thunderstorm conditions. Use that stated backup; do not invent
  one or assume an open-air shelter is an indoor venue.
- If the activity setting or weather information is insufficient to decide,
  assume compatibility, as required by this assignment. Known outdoor
  exposure during rain is sufficient to decide incompatibility.

Judge weather suitability only. Traveler interests, cost, and scheduling
are checked elsewhere. Base the explanation on the supplied facts.

## Output format

Return exactly these two sections, without Markdown fences:

REASONING:
One short explanation of the setting, weather, and any stated indoor backup.

FINAL ANSWER:
One of IS_COMPATIBLE or IS_INCOMPATIBLE, alone on the final line.

Do not write either verdict label in the explanation. Choose one verdict.

## Examples

Activity: Indoor pottery class
Description: A workshop inside the community arts studio.
Weather Condition: thunderstorm
REASONING:
The workshop is indoors and protected from the storm.
FINAL ANSWER:
IS_COMPATIBLE

Activity: Ridge hike
Description: A guided outdoor trail hike with no indoor alternative.
Weather Condition: rainy
REASONING:
The hike is outdoors during rain and has no indoor backup.
FINAL ANSWER:
IS_INCOMPATIBLE

Activity: Outdoor concert with rain venue
Description: The concert is outdoors; if it rains, it moves into the town hall.
Weather Condition: thunderstorm
REASONING:
The description provides an indoor venue for rainy weather.
FINAL ANSWER:
IS_COMPATIBLE

Activity: Community gathering
Description: A neighborhood gathering; no venue details are available.
Weather Condition: rainy
REASONING:
The setting is unspecified, so the assignment's insufficient-information rule applies.
FINAL ANSWER:
IS_COMPATIBLE
""".strip()


def parse_weather_verdict(response: str | None) -> bool:
    """Read the final verdict, rejecting missing or ambiguous output sections."""
    import re

    message = response or ""
    match = re.fullmatch(
        r"\s*REASONING:\s*(.+?)\s*FINAL ANSWER:\s*(IS_COMPATIBLE|IS_INCOMPATIBLE)\s*",
        message,
        flags=re.DOTALL,
    )
    if (
        match is None
        or not match.group(1).strip()
        or message.count("REASONING:") != 1
        or message.count("FINAL ANSWER:") != 1
    ):
        raise RuntimeError(
            "Invalid weather response: expected REASONING and a single FINAL ANSWER verdict."
        )
    return match.group(2) == "IS_COMPATIBLE"


def eval_activities_and_weather_are_compatible(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that no outdoor-only activities are scheduled during inclement weather conditions.

    Args:
        vacation_info (VacationInfo): Contains the vacation details
        final_output (TravelPlan): Contains the daily activities and weather

    Raises:
        AgentError: If any outdoor activities are scheduled during weather conditions that could ruin them
    """
    from project_lib import do_chat_completion

    activities_that_are_incompatible = []

    for itinerary_day in final_output.itinerary_days:
        weather_condition = itinerary_day.weather.condition

        for activity_recommendation in itinerary_day.activity_recommendations:
            resp = do_chat_completion(
                messages=[
                    {
                        "role": "system",
                        "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": f"Activity: {activity_recommendation.activity.name}\nDescription: {activity_recommendation.activity.description}\nWeather Condition: {weather_condition}",
                    },
                ],
                client=client,
                # Vocareum rejects the starter nano model; use the working course mini model.
                model=OpenAIModel.GPT_41_MINI,
            )

            is_compatible = parse_weather_verdict(resp)

            if is_compatible:
                print(
                    f"✅ Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are compatible."
                )

            else:
                activities_that_are_incompatible.append(
                    activity_recommendation.activity.name
                )
                print(
                    f"❌ Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are incompatible."
                )

    if activities_that_are_incompatible:
        raise AgentError(
            f"Activities that may be ruined by inclement weather: {activities_that_are_incompatible}"
        )


eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[
        eval_activities_and_weather_are_compatible
    ],
)

eval_results

✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Artful Athletics: Paint & Play Extravaganza (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity AgentsVille Art & Music Fusion Fest (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity Soundtrack Picnic: Lunchtime Movies & Melodies (on 2025-06-12) and weather 'thunderstorm' are compatible.


✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.


EvaluationResults(success=True, failures=[], eval_functions=['eval_activities_and_weather_are_compatible'])

In [18]:
# Run all of the evaluation functions again
# No changes needed here.

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_daily_schedule_is_valid,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
]

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

eval_results.model_dump()

✅ Traveler Yuri has a match with interest {tennis, cooking} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Artful Athletics: Paint & Play Extravaganza
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art, music} at AgentsVille Art & Music Fusion Fest
✅ Traveler Hiro has a match with interest {music} at Soundtrack Picnic: Lunchtime Movies & Melodies


✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Artful Athletics: Paint & Play Extravaganza (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity AgentsVille Art & Music Fusion Fest (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity Soundtrack Picnic: Lunchtime Movies & Melodies (on 2025-06-12) and weather 'thunderstorm' are compatible.


✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.


{'success': True,
 'failures': [],
 'eval_functions': ['eval_start_end_dates_match',
  'eval_daily_schedule_is_valid',
  'eval_total_cost_is_accurate',
  'eval_itinerary_events_match_actual_events',
  'eval_itinerary_satisfies_interests',
  'eval_total_cost_is_within_budget',
  'eval_activities_and_weather_are_compatible']}

## Defining the Tools

Our ItineraryRevisionAgent will be a ReAct-based agent that will use tools to:
- Evaluate/Re-evaluate the itinerary
- Use a calculator since LLMs sometimes struggle with arithmetic
- Call the activities API to get more information about activities
- Return the final itinerary


In [19]:
# Helper function to generate tool descriptions from function docstrings
# No changes needed here.

def get_tool_descriptions_string(fns):
    """Generates a tool description from a function's docstring.
    Args:
        fns (list): List of functions to generate descriptions for.
    Returns:
        str: A formatted string containing the function names and their descriptions."""
    resp = ""
    for fn in fns:
        function_name = fn.__name__
        function_doc = fn.__doc__ or "No description provided."

        resp += f"* `{function_name}`: {function_doc}\n"

    return resp

In [20]:
# Define the calculator tool that evaluates mathematical expressions.
# No changes needed here.

def calculator_tool(input_expression) -> float:
    """Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    """
    import numexpr as ne
    return float(ne.evaluate(input_expression))


assert calculator_tool("1 + 1") == 2.0

print(get_tool_descriptions_string([calculator_tool]))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    



In [21]:
# Tool to fetch activities for a given date and city.

def get_activities_by_date_tool(date: str, city: str) -> List[dict]:
    """Find available activities when adding or replacing events on a trip day.

    Args:
        date (str): Activity date in YYYY-MM-DD format. The supplied calendar
            covers 2025-06-10 through 2025-06-15, including both dates.
        city (str): Destination city. The supplied calendar supports AgentsVille.

    Returns:
        List[dict]: Complete activity records with IDs, names, ISO date/time
            strings, locations, descriptions, integer prices, and interest tags.
            Returns an empty list when the city or date has no available data.

    Example:
        get_activities_by_date_tool(date="2025-06-12", city="AgentsVille")
    """
    from project_lib import call_activities_api_mocked
    resp = call_activities_api_mocked(date=date, city=city)

    return [Activity.model_validate(activity).model_dump(mode="json") for activity in resp]



assert len(get_activities_by_date_tool("2025-06-10", "AgentsVille")) > 0

print(get_tool_descriptions_string([get_activities_by_date_tool]))

* `get_activities_by_date_tool`: Find available activities when adding or replacing events on a trip day.

    Args:
        date (str): Activity date in YYYY-MM-DD format. The supplied calendar
            covers 2025-06-10 through 2025-06-15, including both dates.
        city (str): Destination city. The supplied calendar supports AgentsVille.

    Returns:
        List[dict]: Complete activity records with IDs, names, ISO date/time
            strings, locations, descriptions, integer prices, and interest tags.
            Returns an empty list when the city or date has no available data.

    Example:
        get_activities_by_date_tool(date="2025-06-12", city="AgentsVille")
    



In [22]:
# Tool to run all evaluation functions on a travel plan.

def run_evals_tool(travel_plan: TravelPlan | dict | None = None) -> dict:
    """Evaluate a complete candidate itinerary against all current trip requirements.

    Args:
        travel_plan (TravelPlan | dict | None): The entire candidate as a JSON
            object matching TravelPlan, not a patch. Omit this argument or use
            null to evaluate the supplied original itinerary.

    Returns:
        dict: A boolean success value and a list of failure messages.
    """
    if travel_plan is None:
        travel_plan = travel_plan_1
    if isinstance(travel_plan, dict):
        travel_plan = TravelPlan.model_validate(travel_plan)

    resp = get_eval_results(
        vacation_info=vacation_info,
        final_output=travel_plan,
        eval_functions=ALL_EVAL_FUNCTIONS,
    )
    return {
        # Show the success status and any failures
        "success": resp.success,
        "failures": resp.failures,
    }


print(get_tool_descriptions_string([run_evals_tool]))

* `run_evals_tool`: Evaluate a complete candidate itinerary against all current trip requirements.

    Args:
        travel_plan (TravelPlan | dict | None): The entire candidate as a JSON
            object matching TravelPlan, not a patch. Omit this argument or use
            null to evaluate the supplied original itinerary.

    Returns:
        dict: A boolean success value and a list of failure messages.
    



In [23]:
# Let's double check that the tool works as expected.
# You should see the same results as before
run_evals_tool(travel_plan=travel_plan_1)

✅ Traveler Yuri has a match with interest {tennis, cooking} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Artful Athletics: Paint & Play Extravaganza
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art, music} at AgentsVille Art & Music Fusion Fest
✅ Traveler Hiro has a match with interest {music} at Soundtrack Picnic: Lunchtime Movies & Melodies


✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Artful Athletics: Paint & Play Extravaganza (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity AgentsVille Art & Music Fusion Fest (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity Soundtrack Picnic: Lunchtime Movies & Melodies (on 2025-06-12) and weather 'thunderstorm' are compatible.


✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.


{'success': True, 'failures': []}

In [24]:
# A tool to return the final travel plan

def final_answer_tool(final_output: TravelPlan) -> TravelPlan:
    """Returns the final travel plan

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    """
    return TravelPlan.model_validate(final_output)


print(get_tool_descriptions_string([final_answer_tool]))

* `final_answer_tool`: Returns the final travel plan

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    



In [25]:
# List of all tools available for the agent
# No changes needed here.

ALL_TOOLS = [
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
]
print(get_tool_descriptions_string(ALL_TOOLS))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    
* `get_activities_by_date_tool`: Find available activities when adding or replacing events on a trip day.

    Args:
        date (str): Activity date in YYYY-MM-DD format. The supplied calendar
            covers 2025-06-10 through 2025-06-15, including both dates.
        city (str): Destination city. The supplied calendar supports AgentsVille.

    Returns:
        List[dict]: Complete activity records with IDs, names, ISO date/time
            strings, locations, descriptions, integer prices, and interest tags.
            Returns an empty list when the city or date has no available data.

    Example:
        get_activities_by_date_tool(date="2025-06

## The ItineraryRevisionAgent

The ItineraryRevisionAgent will
* take initial feedback from the user about the itinerary and
* use the tools defined above

to refine original itinerary iteratively using a ReAct-based approach.

In [26]:
# Get the traveler's feedback and create a new evaluation function to check if the feedback was incorporated.

TRAVELER_FEEDBACK = "I want to have at least two activities per day."



def parse_feedback_verdict(response: str | None) -> tuple[str, str]:
    """Read the final feedback label and reason without searching the analysis."""
    import re

    message = response or ""
    match = re.fullmatch(
        r"\s*ANALYSIS:\s*(.+?)\s*FINAL OUTPUT:\s*(FULLY_INCORPORATED|PARTIALLY_INCORPORATED|NOT_INCORPORATED|UNKNOWN)\s*REASON:\s*(.+?)\s*",
        message, flags=re.DOTALL,
    )
    if (match is None or not match.group(1).strip() or not match.group(3).strip()
            or message.count("FINAL OUTPUT:") != 1):
        raise RuntimeError("Invalid feedback response: expected ANALYSIS, one FINAL OUTPUT label, and REASON.")
    return match.group(2), match.group(3).strip()


def eval_traveler_feedback_is_incorporated(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Checks if the traveler's feedback was incorporated into the revised travel plan.

    Args:
        vacation_info (VacationInfo): The vacation information.
        final_output (TravelPlan): The revised travel plan.

    Raises:
        AgentError: If the traveler's feedback was not successfully incorporated.
    """

    agent = ChatAgent(
        system_prompt="""You are an expert in evaluating whether a travel plan incorporates traveler feedback.

    ## Output Format

    Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:

        ANALYSIS:
        A short summary of the activity count on each trip day.


        FINAL OUTPUT:
        One of FULLY_INCORPORATED, PARTIALLY_INCORPORATED, NOT_INCORPORATED, or UNKNOWN, alone on this line.
        REASON: [reasoning for the final output]

    Judge every trip date. For the supplied feedback, every day needs at least two activities.
    Use verdict labels only in FINAL OUTPUT.
    """,
        client=client,
        model=OpenAIModel.GPT_41,  # Use a powerful model for checking traveler feedback
    )

    resp = agent.chat(
        f"""Traveler Feedback: {TRAVELER_FEEDBACK}
    Revised Travel Plan: {final_output.model_dump_json()}
    """,
    )
    verdict, reason = parse_feedback_verdict(resp)
    missing_days = [str(day.date) for day in final_output.itinerary_days if len(day.activity_recommendations) < 2]
    if verdict != "FULLY_INCORPORATED" or missing_days:
        raise AgentError(
            f"Traveler feedback was not fully incorporated. {reason} "
            f"Days with fewer than two activities: {missing_days}"
        )

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_daily_schedule_is_valid,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
    eval_traveler_feedback_is_incorporated,  # Add this new evaluation
]

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

✅ Traveler Yuri has a match with interest {tennis, cooking} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Artful Athletics: Paint & Play Extravaganza
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art, music} at AgentsVille Art & Music Fusion Fest
✅ Traveler Hiro has a match with interest {music} at Soundtrack Picnic: Lunchtime Movies & Melodies


✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Artful Athletics: Paint & Play Extravaganza (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity AgentsVille Art & Music Fusion Fest (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity Soundtrack Picnic: Lunchtime Movies & Melodies (on 2025-06-12) and weather 'thunderstorm' are compatible.


✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.

╔════════════════════════════════════════════[ ChatAgent - System Prompt ]═════════════════════════════════════════════╗
║ You are an expert in evaluating whether a travel plan incorporates traveler feedback.                                ║
║     ## Output Format                                                                                                 ║
║     Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:                                  ║
║         ANALYSIS:                                                                                                    ║
║         A short summary of the activity count on each trip day.                                                      ║
║         FINAL OUTPUT:                                                                                                ║
║         One of FULLY_INCORPORATED, PARTIALLY_INCORPORAT


╔══════════════════════════════════════════[ ChatAgent - Assistant Response ]══════════════════════════════════════════╗
║ ANALYSIS:                                                                                                            ║
║ - 2025-06-10: 2 activities ("Serve & Savor: Tennis and Taste Luncheon", "Artful Athletics: Paint & Play              ║
║ Extravaganza")                                                                                                       ║
║ - 2025-06-11: 2 activities ("Tech Lunch & Learn: AI Frontiers", "AgentsVille Art & Music Fusion Fest")               ║
║ - 2025-06-12: 2 activities ("Soundtrack Picnic: Lunchtime Movies & Melodies", "Tech & Film Fusion Night")            ║
║ FINAL OUTPUT:                                                                                                        ║
║ FULLY_INCORPORATED                                                                                                   ║
║ REASON: Every day in the revi

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match', 'eval_daily_schedule_is_valid', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [27]:
# Revise the plan through the required thought/action/observation protocol.
from project_lib import print_in_box
from openai import OpenAIError

ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
You are an itinerary revision assistant. Improve the supplied TravelPlan using
traveler feedback and the available tools while preserving the trip constraints.

## Task

1. Your first tool must be run_evals_tool with empty arguments, which evaluates
   the supplied original plan. Include both THOUGHT and ACTION in this first
   response, following the complete example at the end of this prompt. Read
   the failures before choosing changes.
2. Incorporate the traveler feedback below. Use get_activities_by_date_tool
   for days needing additions or replacements. Choose real records, preserve
   every activity field, and place explanations in reasons_for_recommendation.
   For each interest match, use "TravelerName: interest_1, interest_2." and
   include ONLY the intersection of that traveler's listed interests and the
   activity's related_interests. An activity tag is not a traveler preference.
   Put weather or timing explanations in separate strings.
3. Preserve city, dates, forecast values, daily coverage, stated interests,
   nonoverlapping times, and unique activity IDs. Include at least two
   activities on every trip day to satisfy the supplied feedback.
4. Avoid outdoor-only events during rain or thunderstorms. An explicitly
   stated indoor backup is allowed; explain it in the recommendation.
5. Use calculator_tool to check the sum of selected activity prices once each
   across the whole trip. Set total_cost to that result and stay within budget.
6. Call run_evals_tool again with the complete revised candidate. Fix any
   failures and evaluate the new candidate again. A schema-valid plan alone
   is not enough to finish.
7. Only after a successful evaluation, call final_answer_tool with that exact
   evaluated plan under final_output. Do not change it after evaluation.
   The original plan must have been evaluated first, and at least one further
   evaluation must have run before finishing.

## Available Tools

{get_tool_descriptions_string(ALL_TOOLS)}

For subsequent evaluations, supply the complete revised TravelPlan under
travel_plan. All supplied plans are complete objects, not partial updates.
Each call must use the exact documented parameter names.

## THINK-ACT-OBSERVE cycle and Output Format

Return one message containing exactly:

THOUGHT:
A short decision summary explaining the next tool choice and the constraint
being addressed.

ACTION:
{{"tool_name": "calculator_tool", "arguments": {{"input_expression": "15 + 20"}}}}

The ACTION above illustrates the JSON format. Choose the appropriate tool and
arguments for the actual step. Emit one JSON object, without Markdown fences.
Do not emit multiple actions or an OBSERVATION. Python executes the action and
returns OBSERVATION in the next user message. Read it before choosing the next
action. Correct an invalid call after an error observation. Terminate only by
calling final_answer_tool after the successful evaluation of the same plan.

## Context

Traveler feedback:
{TRAVELER_FEEDBACK}

Vacation information:
{vacation_info.model_dump_json(indent=2)}

Weather records:
{json.dumps(weather_for_dates, indent=2)}

TravelPlan schema:
{json.dumps(TravelPlan.model_json_schema(), indent=2)}

Your first response must use this complete format:
THOUGHT:
I will evaluate the original itinerary before choosing changes.
ACTION:
{{"tool_name": "run_evals_tool", "arguments": {{}}}}

Every subsequent response must also include THOUGHT and ACTION. The ACTION
root object always has tool_name and arguments. For a revised evaluation,
put the entire plan in arguments.travel_plan and use run_evals_tool. For
completion, put that exact evaluated plan in arguments.final_output and use
final_answer_tool. A bare TravelPlan at the ACTION root is not a tool call.
""".strip()


class ItineraryRevisionAgent(ChatAgent):
    system_prompt = ITINERARY_REVISION_AGENT_SYSTEM_PROMPT
    tools = ALL_TOOLS

    @staticmethod
    def parse_action(response: str) -> tuple[str, dict]:
        """Require one decision summary and one JSON tool action."""
        import re

        match = re.fullmatch(r"\s*THOUGHT:\s*(.+?)\s*ACTION:\s*(.+)\s*", response, flags=re.DOTALL)
        if (match is None or not match.group(1).strip()
                or response.count("THOUGHT:") != 1 or response.count("ACTION:") != 1):
            raise ValueError("Return one nonempty THOUGHT followed by one ACTION JSON object.")
        try:
            action = json.loads(match.group(2))
        except json.JSONDecodeError:
            from json_repair import repair_json

            action = repair_json(match.group(2), return_objects=True)
        if not isinstance(action, dict) or set(action) != {"tool_name", "arguments"}:
            raise ValueError("ACTION must be an object with tool_name and arguments.")
        if not isinstance(action["tool_name"], str) or not isinstance(action["arguments"], dict):
            raise ValueError("tool_name must be a string and arguments must be an object.")
        return match.group(1).strip(), action

    @staticmethod
    def plan_key(plan: TravelPlan) -> str:
        """Compare validated plan values independently of JSON key order."""
        return json.dumps(plan.model_dump(mode="json"), sort_keys=True)

    def execute_tool(self, action: dict):
        """Resolve only a registered tool and check its argument names."""
        import inspect

        tool = next((tool for tool in self.tools if tool.__name__ == action["tool_name"]), None)
        if tool is None:
            raise ValueError(f"Unknown tool: {action['tool_name']}.")
        inspect.signature(tool).bind(**action["arguments"])
        return tool(**action["arguments"])

    def run_react_cycle(
        self, original_travel_plan: TravelPlan, max_steps: int = 10,
        model: Optional[OpenAIModel] = None, client=None,
    ) -> TravelPlan:
        """Revise with bounded tool use and return only an evaluated candidate."""
        if type(max_steps) is not int or max_steps < 1:
            raise ValueError("max_steps must be a positive integer.")
        original = TravelPlan.model_validate(original_travel_plan)
        original_key = self.plan_key(original)
        self.reset()
        self.trace = []
        initial_evaluated = False
        evaluation_count = 0
        passed_plan_key = None
        self.add_message("user", f"Here is the itinerary for revision:\n{original.model_dump_json()}")

        for step in range(1, max_steps + 1):
            response = self.get_response(model=model, client=client) or ""
            entry = {"step": step, "response": response}
            self.trace.append(entry)
            try:
                thought, action = self.parse_action(response)
                entry.update(thought=thought, action=action)
                name = action["tool_name"]
                if not initial_evaluated and name != "run_evals_tool":
                    raise ValueError("The first action must evaluate the original plan with run_evals_tool.")

                if name == "run_evals_tool":
                    passed_plan_key = None
                    candidate_data = action["arguments"].get("travel_plan")
                    candidate = TravelPlan.model_validate(original if candidate_data is None else candidate_data)
                    candidate_key = self.plan_key(candidate)
                    if not initial_evaluated and candidate_key != original_key:
                        raise ValueError("Evaluate the unchanged original plan before proposing a revision.")
                    evaluated_action = {"tool_name": name, "arguments": {**action["arguments"], "travel_plan": candidate.model_dump(mode="json")}}
                    result = self.execute_tool(evaluated_action)
                    entry["used_original_default"] = candidate_data is None
                    if (not isinstance(result, dict) or type(result.get("success")) is not bool
                            or not isinstance(result.get("failures"), list)):
                        raise ValueError("Evaluation tool must return success as a boolean and failures as a list.")
                    initial_evaluated = True
                    evaluation_count += 1
                    if result["success"] and not result["failures"]:
                        passed_plan_key = candidate_key
                elif name == "final_answer_tool":
                    candidate = TravelPlan.model_validate(action["arguments"]["final_output"])
                    if evaluation_count < 2 or passed_plan_key != self.plan_key(candidate):
                        raise ValueError("Evaluate this exact final candidate successfully after the initial evaluation.")
                    result = TravelPlan.model_validate(self.execute_tool(action))
                    if self.plan_key(result) != self.plan_key(candidate):
                        raise ValueError("The final-answer tool changed the evaluated candidate.")
                    entry.update(status="completed", observation="OBSERVATION: final_answer_tool returned the evaluated plan.")
                    self.add_message("user", entry["observation"])
                    return result
                else:
                    result = self.execute_tool(action)

                observation = "OBSERVATION: " + json.dumps(result, default=str)
                entry.update(status="executed", observation=observation)
            except OpenAIError:
                raise
            except Exception as error:
                observation = f"OBSERVATION: Error: {error}"
                entry.update(status="rejected", observation=observation)
            self.add_message("user", observation)

        raise RuntimeError(f"ReAct cycle did not complete within {max_steps} steps; inspect the agent trace.")


# Preview one message. The actual revision run starts with a fresh conversation.
itinerary_revision_agent = ItineraryRevisionAgent()
resp = itinerary_revision_agent.chat(
    user_message=f"Here is the itinerary for revision: {travel_plan_1.model_dump_json(indent=2)}",
    add_to_messages=False, model=MODEL, client=client,
) or ""
print_in_box(resp, "Raw Response")
ItineraryRevisionAgent.parse_action(resp)
print("Revision preview contains a THOUGHT and one valid ACTION.")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ You are an itinerary revision assistant. Improve the supplied TravelPlan using                                       ║
║ traveler feedback and the available tools while preserving the trip constraints.                                     ║
║ ## Task                                                                                                              ║
║ 1. Your first tool must be run_evals_tool with empty arguments, which evaluates                                      ║
║    the supplied original plan. Include both THOUGHT and ACTION in this first                                         ║
║    response, following the complete example at the end of this prompt. Read                                          ║
║    the failures before choosing changes.                                                                             ║
║ 2. Incorporate the traveler f


╔═══════════════════════════════════════════════════[ Raw Response ]═══════════════════════════════════════════════════╗
║ THOUGHT:                                                                                                             ║
║ I will evaluate the original itinerary before choosing changes.                                                      ║
║ ACTION:                                                                                                              ║
║ {"tool_name": "run_evals_tool", "arguments": {}}                                                                     ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
Revision preview contains a THOUGHT and one valid ACTION.


In [28]:
# Now let's run the ReAct cycle multiple times to get the revised itinerary.
# Note: If this takes more than a few minutes, then the agent may be stuck in a loop.
# Examine the traces to understand where it is failing and see if adjusting the system prompt helps.
# Since LLMs are stochastic, you will get different results each time you run this cell.
# No changes needed here.

itinerary_revision_agent = ItineraryRevisionAgent()
travel_plan_2 = itinerary_revision_agent.run_react_cycle(
    original_travel_plan=travel_plan_1, max_steps=15,
    model=MODEL,
    client=client,
)

print("✅ Revised itinerary generated successfully. Congratulations!")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ You are an itinerary revision assistant. Improve the supplied TravelPlan using                                       ║
║ traveler feedback and the available tools while preserving the trip constraints.                                     ║
║ ## Task                                                                                                              ║
║ 1. Your first tool must be run_evals_tool with empty arguments, which evaluates                                      ║
║    the supplied original plan. Include both THOUGHT and ACTION in this first                                         ║
║    response, following the complete example at the end of this prompt. Read                                          ║
║    the failures before choosing changes.                                                                             ║
║ 2. Incorporate the traveler f


╔═══════════════════════════════════[ ItineraryRevisionAgent - Assistant Response ]════════════════════════════════════╗
║ THOUGHT:                                                                                                             ║
║ I will evaluate the original itinerary before choosing changes to address the traveler's feedback requesting at      ║
║ least two activities per day.                                                                                        ║
║ ACTION:                                                                                                              ║
║ {"tool_name": "run_evals_tool", "arguments": {}}                                                                     ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
✅ Traveler Yuri has a match with interest {tennis, cooking} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {techn

✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Artful Athletics: Paint & Play Extravaganza (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity AgentsVille Art & Music Fusion Fest (on 2025-06-11) and weather 'partly cloudy' are compatible.


... (10 outputs left out)

✅ Activity Soundtrack Picnic: Lunchtime Movies & Melodies (on 2025-06-12) and weather 'thunderstorm' are compatible.


✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.

╔════════════════════════════════════════════[ ChatAgent - System Prompt ]═════════════════════════════════════════════╗
║ You are an expert in evaluating whether a travel plan incorporates traveler feedback.                                ║
║     ## Output Format                                                                                                 ║
║     Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:                                  ║
║         ANALYSIS:                                                                                                    ║
║         A short summary of the activity count on each trip day.                                                      ║
║         FINAL OUTPUT:                                                                                                ║
║         One of FULLY_INCORPORATED, PARTIALLY_INCORPORAT


╔══════════════════════════════════════════[ ChatAgent - Assistant Response ]══════════════════════════════════════════╗
║ ANALYSIS:                                                                                                            ║
║ - 2025-06-10: Two activities ("Serve & Savor: Tennis and Taste Luncheon" and "Artful Athletics: Paint & Play         ║
║ Extravaganza")                                                                                                       ║
║ - 2025-06-11: Two activities ("Tech Lunch & Learn: AI Frontiers" and "AgentsVille Art & Music Fusion Fest")          ║
║ - 2025-06-12: Two activities ("Soundtrack Picnic: Lunchtime Movies & Melodies" and "Tech & Film Fusion Night")       ║
║ FINAL OUTPUT:                                                                                                        ║
║ FULLY_INCORPORATED                                                                                                   ║
║ REASON: Each day in the revis


╔═══════════════════════════════════[ ItineraryRevisionAgent - Assistant Response ]════════════════════════════════════╗
║ THOUGHT:                                                                                                             ║
║ The revised itinerary with two activities on each day meets the traveler's feedback and all constraints, and it      ║
║ passed the evaluation without failures. The total cost is within budget. I will now finalize this travel plan as the ║
║ output.                                                                                                              ║
║ ACTION:                                                                                                              ║
║ {"tool_name": "final_answer_tool", "arguments": {"final_output":{"city":"AgentsVille","start_date":"2025-06-         ║
║ 10","end_date":"2025-06-12","total_cost":103,"itinerary_days":[{"date":"2025-06-                                     ║
║ 10","weather":{"temperature":

In [29]:
# Last let's double check that the revised travel plan passes all evaluation functions.
# No changes needed here.

eval_results_2 = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_2,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

assert eval_results_2.success, f"❌ Read the traces above and modify the system prompt.\n\nFailures: {eval_results_2.failures}"

print("✅ All evaluation functions passed successfully for the revised travel plan.")

eval_results_2

✅ Traveler Yuri has a match with interest {tennis, cooking} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Artful Athletics: Paint & Play Extravaganza
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art, music} at AgentsVille Art & Music Fusion Fest
✅ Traveler Hiro has a match with interest {music} at Soundtrack Picnic: Lunchtime Movies & Melodies


✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Artful Athletics: Paint & Play Extravaganza (on 2025-06-10) and weather 'clear' are compatible.


✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity AgentsVille Art & Music Fusion Fest (on 2025-06-11) and weather 'partly cloudy' are compatible.


✅ Activity Soundtrack Picnic: Lunchtime Movies & Melodies (on 2025-06-12) and weather 'thunderstorm' are compatible.


✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.

╔════════════════════════════════════════════[ ChatAgent - System Prompt ]═════════════════════════════════════════════╗
║ You are an expert in evaluating whether a travel plan incorporates traveler feedback.                                ║
║     ## Output Format                                                                                                 ║
║     Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:                                  ║
║         ANALYSIS:                                                                                                    ║
║         A short summary of the activity count on each trip day.                                                      ║
║         FINAL OUTPUT:                                                                                                ║
║         One of FULLY_INCORPORATED, PARTIALLY_INCORPORAT


╔══════════════════════════════════════════[ ChatAgent - Assistant Response ]══════════════════════════════════════════╗
║ ANALYSIS:                                                                                                            ║
║ 2025-06-10: 2 activities ("Serve & Savor: Tennis and Taste Luncheon" and "Artful Athletics: Paint & Play             ║
║ Extravaganza")                                                                                                       ║
║ 2025-06-11: 2 activities ("Tech Lunch & Learn: AI Frontiers" and "AgentsVille Art & Music Fusion Fest")              ║
║ 2025-06-12: 2 activities ("Soundtrack Picnic: Lunchtime Movies & Melodies" and "Tech & Film Fusion Night")           ║
║ FINAL OUTPUT:                                                                                                        ║
║ FULLY_INCORPORATED                                                                                                   ║
║ REASON: Every day of the trip

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match', 'eval_daily_schedule_is_valid', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [30]:
# Show the final travel plan in a readable format.
# No changes needed here.

from IPython.display import display

for itinerary_day in travel_plan_2.itinerary_days:
    print(f"Date: {itinerary_day.date}")
    print(
        f"Weather: {itinerary_day.weather.condition} ({itinerary_day.weather.temperature}°{itinerary_day.weather.temperature_unit})"
    )

    activities_df = pd.DataFrame(
        [
            activity_recommendation.activity.model_dump()
            for activity_recommendation in itinerary_day.activity_recommendations
        ]
    )
    display(activities_df)

Date: 2025-06-10
Weather: clear (31.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00:00,2025-06-10 13:30:00,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"
1,event-2025-06-10-2,Artful Athletics: Paint & Play Extravaganza,2025-06-10 15:00:00,2025-06-10 17:00:00,"Creative Courts Park, AgentsVille","Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide! At 'Artful Athletics: Paint & Play Extravaganza', you'll participate in collaborative outdoor murals inspired by your favorite sports, and then get moving with fun, friendly sports mini-games. Whether you love painting or playing, this event celebrates creativity, teamwork, and the joy of movement under the open sky. Perfect for art lovers and sports enthusiasts alike—come ready to express yourself and get active! (Event is held outdoors; in case of rain, we move indoors to the Community Gym nearby.)",15,"[art, sports]"


Date: 2025-06-11
Weather: partly cloudy (34.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00:00,2025-06-11 13:30:00,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI creativity).",20,"[technology, music, photography, writing]"
1,event-2025-06-11-2,AgentsVille Art & Music Fusion Fest,2025-06-11 15:00:00,2025-06-11 17:30:00,"The Echo Gardens Amphitheater, AgentsVille","Immerse yourself in an unforgettable afternoon at the Echo Gardens Amphitheater, where the vibrant worlds of art and music collide! Surrounded by lush gardens under the open sky, enjoy live performances from talented local musicians while exploring an interactive outdoor art gallery featuring works from AgentsVille's creative community. This engaging outdoor event is perfect for art and music enthusiasts who love to be inspired and connect with fellow creatives. Don't miss out on the fusion of melodies and colors in a relaxing, friendly atmosphere!",18,"[art, music]"


Date: 2025-06-12
Weather: thunderstorm (28.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-12-1,Soundtrack Picnic: Lunchtime Movies & Melodies,2025-06-12 12:00:00,2025-06-12 13:30:00,"Starlight Amphitheater, AgentsVille","Experience the magic of classic movie scenes paired with live music at the outdoor Starlight Amphitheater! Bring your lunch and relax on the lawn as musicians perform iconic film soundtracks while selected clips light up our open-air screen. Perfect for movie buffs and music lovers alike, this engaging event celebrates both arts in a sunny lunchtime setting. In case of rain, the event will move indoors to the adjacent Harmony Hall. Come for the tunes, stay for the cinematic wonder!",15,"[movies, music]"
1,event-2025-06-12-3,Tech & Film Fusion Night,2025-06-12 19:00:00,2025-06-12 21:30:00,"Virtual Reality Theater, Silicon Plaza, AgentsVille","Dive into an immersive evening where the magic of movies meets the latest in technology! Join fellow movie buffs and tech enthusiasts for a special screening of cutting-edge sci-fi short films, followed by an interactive panel with local filmmakers and VR technologists. Experience the future of entertainment and discuss how technology is transforming the world of cinema. This exciting, indoor event at the Virtual Reality Theater is perfect for anyone interested in technology and movies.",15,"[technology, movies]"


## And, just for fun!

In [31]:
# Summarize the checked itinerary in text; audio is optional in the assignment.
from IPython.display import Markdown, display
from project_lib import do_chat_completion


def summarize_trip(vacation_info, itinerary, evaluation_results, client, model):
    """Generate a short trip summary only after the final checks pass."""
    if not evaluation_results.success:
        raise ValueError("The final itinerary must pass its evaluations before summarizing.")
    summary = do_chat_completion(
        client=client,
        model=model,
        messages=[{
            "role": "user",
            "content": f"""Write a short narrative summary of this AgentsVille trip in four paragraphs,
about 140–180 words. Introduce Yuri and Hiro, the trip dates, and total activity
cost, then describe one day per paragraph in time order. This is a planned trip;
use future tense and treat both travelers as attending the selected itinerary.

Use only the supplied facts. State costs in fictional units, with no currency
symbols. Attribute an interest only to the traveler who actually lists it.
Refer to the travelers by name or as "the travelers" without assuming a
relationship, gender, or pronouns. Explain the specified indoor rain backup.
Use plain, specific wording. Do not invent bookings, transport, extra meals,
preferences, or restrictions. Do not list individual activity prices. Avoid
promotional descriptions and closing generalizations.

Vacation information:
{vacation_info.model_dump_json()}

Final evaluated itinerary:
{itinerary.model_dump_json()}
""",
        }],
    )
    if not isinstance(summary, str) or not summary.strip():
        raise ValueError("The summary model returned no text.")
    return summary.strip()


trip_summary = summarize_trip(vacation_info, travel_plan_2, eval_results_2, client, MODEL)
display(Markdown(trip_summary))


Yuri and Hiro will visit AgentsVille from June 10 to June 12, 2025. Their planned activities will cost a total of 103 units. Both travelers have different interests that shape the itinerary: Yuri enjoys tennis, cooking, comedy, and technology, while Hiro appreciates reading, music, theatre, and art.

On June 10, with clear weather and a temperature of 31 degrees Celsius, Yuri and Hiro will begin their day with "Serve & Savor: Tennis and Taste Luncheon." This event combines tennis and cooking in an outdoor setting at The Grand Racquet Terrace. Later in the afternoon, they will attend "Artful Athletics: Paint & Play Extravaganza" at Creative Courts Park, where they will participate in outdoor mural painting and sports mini-games. This event can move indoors to the Community Gym in case of rain, though clear weather is expected.

On June 11, the weather will be partly cloudy with a temperature of 34 degrees. The travelers will join the indoor "Tech Lunch & Learn: AI Frontiers" at The Digital Atrium, featuring talks and demos about artificial intelligence. In the afternoon, they will attend the outdoor "AgentsVille Art & Music Fusion Fest" at Echo Gardens Amphitheater, where live music performances and an interactive art gallery will take place.

On their final day, June 12, thunderstorms and a cooler temperature of 28 degrees are forecasted. Yuri and Hiro will start with "Soundtrack Picnic: Lunchtime Movies & Melodies" at Starlight Amphitheater, an outdoor event with indoor backup at Harmony Hall due to the weather. In the evening, they will attend the indoor "Tech & Film Fusion Night" at the Virtual Reality Theater, combining technology and movie experiences sheltered from the storm.

## CONGRATULATIONS! 🎉🥳👏

You have successfully planned a stellar vacation to AgentsVille! Your AI travel agent has demonstrated advanced reasoning techniques, including role-based prompting, chain-of-thought reasoning, ReAct prompting, and feedback loops

Give yourself a pat on the back for completing this project and completing this course!